# Root finding with `scipy.optimize.root_scalar`

`root_scalar(f, method=..., ...)` solves $f(x) = 0$ for a scalar $f$. The method is chosen by what you can provide.

| need | method | guaranteed? | order |
|---|---|---|---|
| bracket $[a,b]$ | `bisect` | yes | 1, linear |
| bracket $[a,b]$ | `brentq`, `brenth` | yes | between 1 and 1.62 |
| bracket $[a,b]$ | `ridder` | yes | 2.0 |
| bracket $[a,b]$ | `toms748` | yes | 2.7 |
| $x_0$, $x_1$ | `secant` | no | 1.62 |
| $x_0$, $f'$ | `newton` | no | 2.0 |
| $x_0$, $f'$, $f''$ | `halley` | no | 3.0 |

Table adapted from the SciPy documentation, *Optimization and root finding → Root finding → Scalar functions*:
<https://docs.scipy.org/doc/scipy/reference/optimize.html#module-scipy.optimize>

- **Bracketing** methods always converge. Bisection gains one bit per step; the others converge faster.
- **Derivative-based** methods (built on Newton) are fast only if $x_0$ is close to the root. They also work for complex $x$.
- The orders are for a simple root ($f'(r) \neq 0$) and describe the final steps, close to the root.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

## Example 1: bracketing methods on $f(x) = x^{10} - 1$, bracket $[0, 1.3]$

In class, false position needed 61 steps here and plain Newton jumped to $x_1 \approx 5.4$.
Every bracketing method is guaranteed to converge.

In [2]:
f = lambda x: x**10 - 1

sol = root_scalar(f, method="bisect", bracket=[0, 1.3], xtol=1e-12)
print("root       =", sol.root)
print("iterations =", sol.iterations)
print("converged  =", sol.converged)

root       = 1.0000000000002272
iterations = 41
converged  = True


The result is a `RootResults` object. The fields you will use most:

- `sol.root`: the approximate root
- `sol.iterations`: number of steps
- `sol.converged`: `True` if the stopping test was met. **Always check it**: a failed run still returns a `root`.

In [3]:
for m in ["bisect", "brentq", "brenth", "ridder", "toms748"]:
    sol = root_scalar(f, method=m, bracket=[0, 1.3], xtol=1e-12)
    print(f"{m:8s} root = {sol.root:.15f}   iterations = {sol.iterations:2d}   converged = {sol.converged}")

bisect   root = 1.000000000000227   iterations = 41   converged = True
brentq   root = 0.999999999999847   iterations =  9   converged = True
brenth   root = 1.000000000000001   iterations =  9   converged = True
ridder   root = 1.000000000000500   iterations =  6   converged = True
toms748  root = 1.000000000000000   iterations =  6   converged = True


- `bisect` is the slowest; its step count follows $\log_2\frac{b-a}{\text{tol}}$ from class.
- The others combine fast interpolation steps with safe steps inside the bracket.
- `brentq` is the default when you pass only `bracket=`: a good first choice whenever $f$ changes sign.

## Example 2: open methods on $f(x) = \cos x - x$, start $x_0 = 1$

No bracket: these methods need starting points, and Newton and Halley need derivatives.

In [4]:
g   = lambda x: np.cos(x) - x
dg  = lambda x: -np.sin(x) - 1
d2g = lambda x: -np.cos(x)

s1 = root_scalar(g, method="secant", x0=1.0, x1=0.5)
s2 = root_scalar(g, method="newton", x0=1.0, fprime=dg)
s3 = root_scalar(g, method="halley", x0=1.0, fprime=dg, fprime2=d2g)

for s in [s1, s2, s3]:
    print(f"{s.method:8s} root = {s.root:.15f}   iterations = {s.iterations}   converged = {s.converged}")

secant   root = 0.739085133215160   iterations = 5   converged = True
newton   root = 0.739085133215161   iterations = 4   converged = True
halley   root = 0.739085133215161   iterations = 3   converged = True


Higher order means fewer iterations: Halley < Newton < secant.

A derivative-based method can fail. From $x_0 = 0$, Newton on $x^3 - 2x + 2$ cycles $0, 1, 0, 1, \ldots$:

In [5]:
h  = lambda x: x**3 - 2*x + 2
dh = lambda x: 3*x**2 - 2

sol = root_scalar(h, method="newton", x0=0.0, fprime=dh, maxiter=50)
print("root =", sol.root, "  converged =", sol.converged, "  flag =", sol.flag)
print("f(root) =", h(sol.root))

root = 0.0   converged = False   flag = convergence error
f(root) = 2.0


## Example 3: a complex root, $f(z) = z^2 + 1$

$z^2 + 1$ has no real root, so there is no sign change and no bracket. Its roots are $\pm i$.
`secant`, `newton` and `halley` also work for complex $z$: give a **complex** starting point (Python writes $i$ as `1j`).

In [ ]:
f  = lambda z: z**2 + 1
df = lambda z: 2*z

for z0 in [1 +1j , 1 - 1j]:
    sol = root_scalar(f, method="newton", x0=z0, fprime=df)
    print(f"x0 = {z0}:  root = {sol.root:.12f}   iterations = {sol.iterations}   converged = {sol.converged}")

sol = root_scalar(f, method="secant", x0=1 + 1j, x1=0.5 + 0.5j)
print(f"secant:          root = {sol.root:.12f}   iterations = {sol.iterations}   converged = {sol.converged}")

x0 = (1+0.5j):  root = 0.000000000000+1.000000000000j   iterations = 7   converged = True
x0 = (1-1j):  root = 0.000000000000-1.000000000000j   iterations = 6   converged = True
secant:          root = -0.000000000000+1.000000000000j   iterations = 8   converged = True


- The starting point decides which root is found: above the real axis gives $+i$, below gives $-i$.
- With a **real** start, every Newton step stays real, so a complex root can never be reached. From $x_0 = 1$ the next iterate is $0$, where $f'(0) = 0$:

In [7]:
sol = root_scalar(f, method="newton", x0=1.0, fprime=df)
print("root =", sol.root, "  converged =", sol.converged, "  flag =", sol.flag)

root = 0.0   converged = False   flag = convergence error


## Exercises

**Syntax for each method**

| method | call |
|---|---|
| `bisect` | `root_scalar(f, method="bisect", bracket=[a, b])` |
| `brentq` | `root_scalar(f, method="brentq", bracket=[a, b])` |
| `brenth` | `root_scalar(f, method="brenth", bracket=[a, b])` |
| `ridder` | `root_scalar(f, method="ridder", bracket=[a, b])` |
| `toms748` | `root_scalar(f, method="toms748", bracket=[a, b])` |
| `secant` | `root_scalar(f, method="secant", x0=x0, x1=x1)` |
| `newton` | `root_scalar(f, method="newton", x0=x0, fprime=df)` |
| `halley` | `root_scalar(f, method="halley", x0=x0, fprime=df, fprime2=d2f)` |
| complex root | same as `secant` / `newton` / `halley`, with a complex start, e.g. `x0=1+1j` |

<br><br>


| optional argument | meaning |
|---|---|
| `xtol`, `rtol` | absolute and relative tolerance on $x$ |
| `maxiter` | maximum number of iterations |


<br><br>

| result field | meaning |
|---|---|
| `sol.root` | approximate root |
| `sol.iterations` | number of iterations |
| `sol.converged` | `True` if the stopping test was met |
| `sol.flag` | reason for stopping |

<br><br>

| plotting | call |
|---|---|
| create a figure and axes | `fig, ax = plt.subplots()` |
| horizontal line $y = c$ (e.g. the $x$-axis, $c = 0$) | `ax.axhline(c, color="k", lw=0.8)` |
| vertical line $x = c$ (e.g. mark a root or bracket end) | `ax.axvline(c, color="r", ls="--")` |




**Exercise 1.** $f(x) = x^3 - 6x^2 + 11x - 6.1$ has three real roots.
Plot $f$ on $[0, 4]$, choose a bracket for each root, and find all three with `bisect`, `brentq` and `toms748`.
Make a table of iterations per method. Which method needs the fewest iterations?

In [8]:
# Exercise 1

**Exercise 2.** $f(x) = \arctan x$ has the root $r = 0$.

(a) Run `newton` and `halley` from $x_0 = 1.3$ and from $x_0 = 1.5$. Report `root`, `converged` and `flag` for each.

(b) Explain the difference between the two starting points.

(c) Find the root reliably with a bracketing method. What bracket did you use, and why is it valid?

In [9]:
# Exercise 2